# The package and its equation language {#sec-package}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/a02-package.ipynb)

Every chapter runs its models with `loopbricks`, a small Python package written for this book. The chapters introduce its pieces as they need them; this appendix puts them all in one place. Use it as a reference when a chapter calls something you have not met, and work through it once if you want to build your own models with the package.

**Who it is for.** Anyone using the package beyond the examples in the chapters. **Prerequisites.** The Python of @sec-python and the stocks, flows and Euler steps of @sec-push-back. **How long.** An hour to read and run; the quick reference in @sec-pkg-reference is enough once you know the ideas.

The package and the LoopBricks web app share the same model files and the same arithmetic, so a model gives the same numbers in both, to the last digit. It simulates with Euler's method by default, exactly as chapter 1 does by hand.

In [ ]:
# Setup: run this cell first. In Colab it installs loopbricks (ten seconds).
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/"
                           "loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print("loopbricks", lb.__version__)

## 1. Molecules: ready-made models

A **molecule** is a small model of one recurring structure, such as a stock draining in proportion to itself or a pipeline of orders. The package ships 73 of them, our own implementations of the structures Hines describes in *Molecules of Structure* [@hines_molecules] plus two archetypes of our own. `lb.molecules()` lists their keys and `lb.molecule(key)` loads one as a `Model`.

In [ ]:
print(len(lb.molecules()), "molecules, for example", lb.molecules()[:6])
m = lb.molecule("decay")
print(m)                      # name, size and the constants you can set
print()
print(m.equations())

`equations()` prints the model the way modellers write it. A stock is shown in `INTEG` form: `INTEG(inflows − outflows, initial value)` means "start at the initial value and add the net flow at every step". In equations, names are written with underscores (`Decay_time`); in the printed list of variables they keep their spaces.

A model also tells you about itself:

| Attribute | What it holds |
|---|---|
| `m.name`, `m.time_unit`, `m.T`, `m.dt` | the title, the unit of time, the default length of a run and the default time step |
| `m.params` | the constants you can set, by Python name, with their default values |
| `m.ranges` | the slider range the app uses for each constant (`min`, `max`, `step`) |
| `m.presets` | named settings, as a list of `{"label": ..., "params": ...}`; `m.preset(label)` returns one as a dictionary |
| `m.stocks`, `m.flows`, `m.vars` | the elements, as lists of dictionaries (see section 6) |

In [ ]:
print(m.time_unit, m.T, m.dt)
print(m.params)
print(m.ranges)
print([p["label"] for p in m.presets], m.preset("Fast decay"))

**Python names.** A constant called `Decay time` in the model is `decay_time` in Python: lower case, with spaces turned into underscores. `lb.name_key("Decay time")` does that conversion. Wherever the package asks for a name it accepts either form and ignores case, so `"Decay time"`, `"decay_time"` and `"DECAY TIME"` all find the same element.

## 2. Running a model

`run()` simulates and returns a pandas DataFrame with one row per time step and one column per stock, flow and variable. The index is time, named with its unit.

In [ ]:
df = m.run()
print(df.index.name, "| rows:", len(df), "| columns:", list(df.columns))
df.head(3)

Its options, all optional:

| Option | Meaning | Example |
|---|---|---|
| `T`, `dt` | length of the run and time step, overriding the model's own | `run(T=20, dt=0.05)` |
| any constant by Python name | set that constant for this run only | `run(decay_time=2)` |
| `preset` | apply a named preset first; keywords then override it | `run(preset="Fast decay")` |
| `include` | `"all"` (default), `"stocks"`, or a list of names | `run(include=["Stock"])` |
| `columns` | `"name"` (default) or `"key"` for Python-style column names | `run(columns="key")` |
| `inputs` | replace a flow or variable by a series or function of time (section 3) | `run(inputs={"Demand": s})` |
| `method` | `"euler"` (default, as the app) or `"rk4"` (fourth-order Runge-Kutta) | `run(method="rk4")` |
| `warn_clips` | warn when a floor at zero holds a stock up (section 7) | `run(warn_clips=False)` |

The model itself never changes: every option applies to one run.

In [ ]:
fast = m.run(decay_time=2, T=10, include="stocks")
print(fast["Stock"].iloc[-1])                        # after 10 years
print(m.run(preset="Fast decay", T=10)["Stock"].iloc[-1])   # the same setting
exact = 100 * np.exp(-10 / 2)
for method in ("euler", "rk4"):
    end = m.run(decay_time=2, T=10, method=method)["Stock"].iloc[-1]
    print(f"{method:>5}: {end:.6f}  (exact {exact:.6f})")

Euler with the model's step of 0.1 years is close to the exact answer, and the fourth-order Runge-Kutta method is closer still at the same step. The chapters use Euler because it is what the app uses and what chapter 1 explains; `rk4` is there for checking that a result does not depend on the method.

The DataFrame also carries two dictionaries in `df.attrs`: `units`, the unit of each column where the model gives one, and `clipped`, explained in section 7.

**Try it 2.**

- *Check.* Run `decay` for 30 years with a decay time of 10 and print the stock at the end.
- *Apply.* Load `growth`, print its equations and parameters, and find the year its population first exceeds 200.
- *Extend.* Halve `dt` for `growth` three times and print the final population each time. Does it settle?

<details><summary>Worked answer</summary>

```python
print(m.run(T=30, decay_time=10)["Stock"].iloc[-1])
g = lb.molecule("growth")
print(g.equations()); print(g.params)
pop = g.run()["Population"]
print(pop.index[pop > 200][0])          # the first time label where the mask is True
for dt in (g.dt, g.dt / 2, g.dt / 4, g.dt / 8):
    print(dt, g.run(dt=dt)["Population"].iloc[-1])   # creeps up towards the exact value
```

</details>

## 3. Feeding a model from outside: inputs

A constant is one number for the whole run. To drive a model with something that changes over time, such as observed referrals or a forecast, pass `inputs`: a dictionary from the name of a flow or variable to either a pandas Series indexed by time or a Python function of time. A Series is joined with straight lines between its points and held flat beyond its ends.

In [ ]:
sm = lb.molecule("stockmgmt")
demand = pd.Series([10, 10, 14, 14], index=[0, 5, 5.0001, 60])   # a step at week 5
run = sm.run(inputs={"Demand": demand}, T=40)
print(run[["Demand", "Inventory", "Orders"]].iloc[[0, 20, 60, 120, 160]].round(2))

wave = sm.run(inputs={"Demand": lambda t: 10 + 2 * np.sin(2 * np.pi * t / 20)}, T=40)
print(round(wave["Orders"].max(), 2), "peak orders for demand peaking at 12")

An input replaces the element's own equation for that run. Only flows and variables can be replaced; a stock always comes from its flows. Setting the same element both as a constant and as an input is an error, because the package could not know which you meant.

## 4. Many runs at once: run_batch

Sensitivity analysis, calibration and policy grids need hundreds of runs. `run_batch(settings)` takes a DataFrame (or a list of dictionaries) with one row per run and a column per constant, and computes all the runs together with NumPy arrays. Each run gives exactly the numbers `run()` would give with the same settings, much faster.

In [ ]:
settings = pd.DataFrame({"supply_line_weight": [0, 0.5, 1],
                         "acquisition_delay": [6, 6, 6]},
                        index=["ignores pipeline", "half weight", "full weight"])
out = sm.run_batch(settings, T=60)
print(out.head())
print(out.shape, "rows: run x time x variable")

The result is in **long** (tidy) form: one row per run, time and variable, with the columns `run`, `time`, `variable` and `value`. Long form keeps any number of runs in one table and joins back to the settings by `run`. To get the familiar wide table, one column per run, select a variable and **pivot** it: `index` becomes the rows, `columns` the columns and `values` the cells.

In [ ]:
#| fig-alt: "Inventory over 60 weeks rising from 60 towards its target of 100 for three supply line weights: ignoring the pipeline overshoots most, full weight least."
inv = out[out.variable == "Inventory"].pivot(index="time", columns="run",
                                             values="value")
print(inv.iloc[[0, 40, 120, 240]].round(1))
print("\nhighest inventory:", inv.max().round(1).to_dict())

fig, ax = plt.subplots()
inv.plot(ax=ax)
ax.set(xlabel="time (weeks)", ylabel="Inventory (units)")
plt.show()

Figure: inventory in the Stock management molecule for three weights on the supply line, from one call to `run_batch`. The less the ordering rule counts what is already on order, the further inventory overshoots its target of 100.

`include` defaults to `"stocks"` in `run_batch` to keep the table small; pass `include="all"` for every variable. Statistics by group work directly on the long table, for example `out.groupby(["run", "variable"])["value"].max()`.

**Try it 4.**

- *Check.* Run `stockmgmt` for acquisition delays of 2, 4, 6, 8 and 10 weeks with the supply line ignored, and print the highest inventory for each.
- *Apply.* Build the settings with a list of dictionaries instead of a DataFrame. What are the run labels then?
- *Extend.* Use `np.linspace` and `np.meshgrid` to make a 10 by 10 grid of acquisition delay and supply line weight, run it in one batch, and find the share of settings in which inventory ever rises above 110.

<details><summary>Worked answer</summary>

```python
grid = pd.DataFrame({"acquisition_delay": [2, 4, 6, 8, 10], "supply_line_weight": 0})
o = sm.run_batch(grid, T=60)
print(o[o.variable == "Inventory"].groupby("run")["value"].max())
o2 = sm.run_batch([{"acquisition_delay": 2}, {"acquisition_delay": 10}], T=60)
print(o2.run.unique())                  # 0 and 1: the positions in the list
d, w = np.meshgrid(np.linspace(2, 10, 10), np.linspace(0, 1, 10))
big = pd.DataFrame({"acquisition_delay": d.ravel(), "supply_line_weight": w.ravel()})
high = sm.run_batch(big, T=60).query("variable == 'Inventory'").groupby("run")["value"].max()
print((high > 110).mean())   # .query filters rows with a condition written as text
```

</details>

## 5. Changing a model: edited copies

Sometimes a constant is not enough: you want a different equation, a different starting value, or new defaults to hand on to someone else. Three methods return an **edited copy** and leave the original alone, so you can keep both and compare them.

| Method | Changes | Example |
|---|---|---|
| `with_equation(name, equation)` | the equation of a flow or variable | `sm.with_equation("Demand", "10 + STEP(4, 5)")` |
| `with_init(stock, value)` | where a stock starts, a number or an equation | `sm.with_init("Inventory", 80)` |
| `with_params(**defaults)` | the default values of constants | `sm.with_params(supply_line_weight=0)` |

In [ ]:
stepped = sm.with_equation("Demand", "10 + STEP(4, 5)")
low_start = sm.with_init("Inventory", 80)
naive = sm.with_params(supply_line_weight=0)
for label, model in [("original", sm), ("demand steps up", stepped),
                     ("starts low", low_start), ("ignores pipeline", naive)]:
    r = model.run(T=60)
    print(f"{label:>17}: inventory from {r['Inventory'].iloc[0]:.0f}, "
          f"highest {r['Inventory'].max():.1f}; peak orders {r['Orders'].max():.1f}")
print("the original is unchanged:", sm.params["supply_line_weight"])

Making a variable that is worked out from others into something you can set works the same way: `with_equation("Desired supply line", "60")` turns it into a constant, which then appears in `params`.

## 6. Building your own model: the spec

A model is described by a dictionary, the **spec**, which `lb.Model.from_spec(spec)` turns into a `Model`. The same format is what the app's model builder exports, so `lb.Model.from_file("my-model.json")` loads a model drawn in the app. The parts:

| Key | Meaning |
|---|---|
| `name`, `timeUnit`, `T`, `dt` | title, unit of time, default run length and time step |
| `stocks` | list of stocks, each `{"id", "name", "init"}`, optionally `"units"` and `"nonNegative": True` |
| `flows` | list of flows, each `{"id", "name", "eq"}` with `"from"` (the stock it drains) and/or `"to"` (the stock it fills), by stock `id`; leave one out for a flow to or from outside the model |
| `vars` | list of variables, each `{"id", "name", "eq"}`; a plain number makes a constant you can set in `run()` |

Every `id` is a short label of your choice, used only to connect flows to stocks; every `name` is what you see and what equations use. Initial values (`init`) and equations (`eq`) are written in the equation language of section 8, as strings.

In [ ]:
spec = {
    "name": "Clinic list", "timeUnit": "month", "T": 36, "dt": 0.25,
    "stocks": [{"id": "wl", "name": "Waiting list", "init": "Target_list * 1.2",
                "units": "patients", "nonNegative": True}],
    "flows": [
        {"id": "in", "name": "Referrals", "to": "wl",
         "eq": "Base_referrals + STEP(200, 12)", "units": "patients/month"},
        {"id": "out", "name": "Treatments", "from": "wl",
         "eq": "MIN(Capacity, Waiting_list / Minimum_time)", "units": "patients/month"},
    ],
    "vars": [
        {"id": "br", "name": "Base referrals", "eq": "4000", "units": "patients/month"},
        {"id": "cap", "name": "Capacity", "eq": "Base_referrals + (Waiting_list - Target_list) / Correction_time",
         "units": "patients/month"},
        {"id": "tl", "name": "Target list", "eq": "20000", "units": "patients"},
        {"id": "ct", "name": "Correction time", "eq": "6", "units": "month"},
        {"id": "mt", "name": "Minimum time", "eq": "0.5", "units": "month"},
    ],
}
clinic = lb.Model.from_spec(spec)
print(clinic.equations())
print("\nyou can set:", clinic.params)
res = clinic.run()
print(res[["Waiting list", "Capacity"]].iloc[[0, 48, 96, -1]].round(0))
print(res.attrs["units"])

The package checks a spec as it loads: every name in an equation must exist, two elements cannot share a name, flows must point at real stocks, and variables cannot depend on each other in a circle with no stock in between (a stock breaks every loop, because its value at a step is already known). Variables may be listed in any order; the package works out the order to compute them in. A stock may start from an equation, as `Waiting list` does here.

The richer form the molecules use wraps the model with a title, a description, slider ranges and presets: `{"format": "loopbricks-model", "version": 1, "model": {...}, "params": {...}, "presets": [...]}`. Look at `lb.molecule("decay").spec` for an example.

**Try it 6.**

- *Check.* Run `clinic` with a correction time of 2 months. Does the list reach its target faster?
- *Apply.* Add a variable `Extra clinics` with value 0 and include it in `Capacity`, then run it at 300.
- *Extend.* Introduce a mistake on purpose (an equation that names `Capacityy`) and read the error message.

<details><summary>Worked answer</summary>

```python
print(clinic.run(correction_time=2)["Waiting list"].iloc[[0, 24, -1]].round(0))
spec2 = {**spec, "vars": spec["vars"] + [{"id": "ex", "name": "Extra clinics", "eq": "0"}]}
spec2["vars"] = [dict(v) for v in spec2["vars"]]
spec2["vars"][1]["eq"] += " + Extra_clinics"
print(lb.Model.from_spec(spec2).run(extra_clinics=300)["Waiting list"].iloc[-1])
bad = {**spec, "flows": [spec["flows"][0], {**spec["flows"][1], "eq": "Capacityy"}]}
try:
    lb.Model.from_spec(bad)
except lb.LoopBricksError as e:
    print(e)            # Treatments: unknown name "capacityy"
```

</details>

## 7. Floors at zero

Some stocks cannot go below zero: patients, stock on shelves, fish. Marking a stock `"nonNegative": True` puts a **floor** under it: if an outflow would take it below zero in a step, the stock is set to zero instead. That keeps it physical, but the flows and the stock no longer add up exactly in that step, so the package says so. It warns once per run, and records the first time each stock was held up in `df.attrs["clipped"]`.

The better fix is usually in the model: an outflow that cannot exceed what is there, such as `MIN(Capacity, Waiting_list / Minimum_time)` above, which drains a short list gradually and never needs the floor. Chapter 5 discusses both.

In [ ]:
import warnings
blunt = clinic.with_equation("Treatments", "5000")          # treats 5,000 whatever is left
with warnings.catch_warnings(record=True) as caught:        # collect warnings here
    warnings.simplefilter("always")
    r = blunt.run(base_referrals=0, T=12)
print("floor used:", r.attrs["clipped"])
print(caught[0].message if caught else "no warning")
gentle = clinic.with_equation("Treatments", "MIN(5000, Waiting_list / Minimum_time)")
r2 = gentle.run(base_referrals=0, T=12)
print("with the MIN rule:", r2.attrs["clipped"], "| list at month 12:",
      round(r2["Waiting list"].iloc[-1], 1))

`warnings.catch_warnings(record=True)` collects warnings into a list instead of printing them, so we can show the message; normally you would just see it under the cell. `run(warn_clips=False)` hides it when you have already checked that the floor is doing what you intend.

## 8. The equation language

Equations are short formulas in a syntax close to Vensim's and Stella's. The package compiles each one once, then evaluates it at every step.

**Names.** Names ignore case, and spaces and underscores count as the same, so `Waiting_list`, `waiting list` and `WAITING_LIST` are one name. In a formula you usually write underscores, because a space would separate two names; a name in double quotes may contain spaces: `"Waiting list" / 2`. A number cannot run straight into a name: write `2 * Capacity`, not `2Capacity`.

**Arithmetic and comparisons.**

| Syntax | Meaning |
|---|---|
| `+ - * /` | arithmetic; `-x` negates |
| `^` | power: `1.05 ^ 10`; `-2 ^ 2` is `-(2 ^ 2)` |
| `( )` | grouping |
| `< <= > >= = <>` | comparisons, giving 1 for true and 0 for false; `<>` means not equal |

**Functions.**

| Function | Gives | Example |
|---|---|---|
| `MIN(a, b, ...)`, `MAX(a, b, ...)` | smallest or largest of two or more values | `MAX(0, Gap / Time)` |
| `ABS(x)`, `SQRT(x)`, `EXP(x)`, `LN(x)` | absolute value, square root, $e^x$, natural log | `EXP(-TIME / 5)` |
| `IF(c, a, b)` | `a` when `c` is not zero (true), else `b` | `IF(Stock > 100, 50, 0)` |
| `STEP(h, t0)` | 0 before time `t0`, then `h` | `4000 + STEP(200, 12)` |
| `PULSE(h, t0, w)` | `h` from `t0` for a width `w`, else 0; a width of 0 lasts one step | `PULSE(500, 24, 1)` |
| `RAMP(s, t0)` | 0 before `t0`, then rising at slope `s` per time unit | `RAMP(10, 12)` |
| `TABLE(x, x1, y1, x2, y2, ...)` | a lookup: straight lines between the points, flat beyond the ends; x values must increase and points must be plain numbers | `TABLE(Pressure, 0, 1, 1, 1.2, 2, 1.3)` |
| `TIME`, `DT` | the current time and the time step | `IF(TIME < 12, 1, 0)` |

The quickest way to see what a formula does is to run it in a one-variable model:

In [ ]:
#| fig-alt: "Five test signals over 40 months: a step at 10, a pulse from 20 to 22, a ramp from 10, a lookup that rises and levels off, and an IF switch at 30."
def show(**equations):
    """Run each equation as a variable over 40 time units and return the columns."""
    spec = {"name": "Signals", "T": 40, "dt": 0.25, "stocks": [], "flows": [],
            "vars": [{"id": k, "name": k, "eq": eq} for k, eq in equations.items()]}
    return lb.Model.from_spec(spec).run()

sig = show(step="STEP(2, 10)", pulse="PULSE(3, 20, 2)", ramp="RAMP(0.1, 10)",
           lookup="TABLE(TIME, 0, 0, 10, 0.5, 20, 2.5, 30, 3)",
           switch="IF(TIME >= 30, 1.5, 0)")
print(sig.loc[[0, 10, 21, 25, 35]].round(2))
fig, ax = plt.subplots()
sig.plot(ax=ax)
ax.set(xlabel="time", ylabel="value")
plt.show()

Figure: the test signals of the equation language over 40 time units. The lookup is flat beyond its last point at time 30, and the pulse lasts exactly its width of 2.

**Edge cases.** Arithmetic follows the app's JavaScript rules so both give identical numbers: dividing by zero gives infinity, and a root or log of a negative number is not a number. Either one stops a run with a message naming the element and the time, because a model that divides by zero has a mistake in it.

In [ ]:
try:
    show(bad="1 / (TIME - 10)").head()
except lb.LoopBricksError as e:
    print(e)

**Try it 8.**

- *Check.* What does `PULSE(5, 10, 0)` give with `dt = 0.25`, and for how long?
- *Apply.* Write one equation that is 100 before time 12 and 80 afterwards, in two different ways.
- *Extend.* A lookup's points must be numbers. How would you make the lookup's height depend on a constant `Scale`?

<details><summary>Worked answer</summary>

```python
p = show(p="PULSE(5, 10, 0)")["p"]
print(p[p > 0])                          # 5 for one step, at time 10
print(show(a="100 - STEP(20, 12)", b="IF(TIME < 12, 100, 80)").loc[[11.75, 12]])
# multiply outside the lookup: Scale * TABLE(x, 0, 0, 1, 1, 2, 1.5)
```

</details>

## 9. Errors and how to read them

Every error the package raises on purpose is a `lb.LoopBricksError` with a message meant for people: it names the element, says what is wrong and, for a misspelt name, suggests the closest one. Catch `lb.LoopBricksError` to handle them all. The most common:

In [ ]:
for attempt in [lambda: sm.run(supply_line_wieght=0),
                lambda: sm.run(desired_supply_line=50),
                lambda: sm.run(preset="Stable"),
                lambda: lb.molecule("stock_management")]:
    try:
        attempt()
    except lb.LoopBricksError as e:
        print(type(e).__name__, ":", str(e)[:150], "\n")

The first is a typo, the second tries to set a variable that is worked out from others (the message says how to replace it instead), the third names a preset loosely and the fourth asks for a molecule that does not exist. Each message gives the choices.

**Plotting shortcut.** `m.plot()` runs the model and plots its stocks; `m.plot(df, columns=[...])` plots chosen columns of a run you already have. The chapters draw their own figures with matplotlib to label them fully, but `plot` is handy while exploring.

## 10. Quick reference {#sec-pkg-reference}

| Task | Code |
|---|---|
| list and load molecules | `lb.molecules()`, `m = lb.molecule("stockmgmt")` |
| look at a model | `print(m)`, `print(m.equations())`, `m.params`, `m.presets`, `m.ranges` |
| run it | `df = m.run()`, `m.run(T=60, dt=0.125)`, `m.run(preset="...")`, `m.run(supply_line_weight=0)` |
| choose columns | `m.run(include="stocks")`, `m.run(include=["Inventory"])`, `m.run(columns="key")` |
| drive it from outside | `m.run(inputs={"Demand": series_or_function})` |
| many runs | `out = m.run_batch(settings_df)`; `out[out.variable == "X"].pivot(index="time", columns="run", values="value")` |
| edited copies | `m.with_equation("Demand", "10 + STEP(4, 5)")`, `m.with_init("Inventory", 80)`, `m.with_params(...)` |
| build your own | `lb.Model.from_spec({...})`, `lb.Model.from_file("model.json")` |
| check floors | `df.attrs["clipped"]`, `m.run(warn_clips=False)` |
| compare methods | `m.run(method="rk4")` |
| Python name of a model name | `lb.name_key("Decay time")` gives `"decay_time"` |
| catch package errors | `except lb.LoopBricksError as e:` |
| equation language | `+ - * / ^`, comparisons, `MIN MAX ABS SQRT EXP LN IF STEP PULSE RAMP TABLE TIME DT` |